#  Hyperparameter Tuning with GridSearchCV
### 

In [2]:
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score


In [3]:
X = joblib.load("X_full.pkl")
y = joblib.load("y_full.pkl")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


In [4]:
#Logistic Regression Tuning

In [5]:
param_grid_log = {
    "C": [0.01, 0.1, 0.5, 1, 5, 10],
    "solver": ["liblinear"]
}
# liblinear mmodelin matematiksel denklemi çözmesi için kullanıacak olan algoritmanın adı 

In [6]:
log_model = LogisticRegression(max_iter=1000)

grid_log = GridSearchCV(
    log_model,
    param_grid_log,
    cv=5,
    scoring="roc_auc"
)

grid_log.fit(X_train, y_train)


,estimator,LogisticRegre...max_iter=1000)
,param_grid,"{'C': [0.01, 0.1, ...], 'solver': ['liblinear']}"
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [7]:
#Lojistik regresyon modelim için en iyi 'C' değeri hangisidir?

In [8]:
grid_log.best_params_


{'C': 0.1, 'solver': 'liblinear'}

In [9]:
grid_log.best_score_


np.float64(0.794944998176384)

In [10]:
# en iyi hyperparameterları tek değişkene atadım.
best_log = grid_log.best_estimator_

In [11]:
y_pred_log = best_log.predict(X_test)
y_proba_log = best_log.predict_proba(X_test)[:,1]

accuracy_score(y_test, y_pred_log),
roc_auc_score(y_test, y_proba_log)
# Eğitim verisi üzerindeki tahminler
y_pred_train = best_log.predict(X_train)

# Karşılaştırma
print(f"Eğitim Doğruluğu: {accuracy_score(y_train, y_pred_train)}")
print(f"Test Doğruluğu: {accuracy_score(y_test, y_pred_log)}")

Eğitim Doğruluğu: 0.7851002865329513
Test Doğruluğu: 0.7557251908396947


In [12]:
# Random Forest Tuning

In [13]:
param_grid_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10]
}


In [14]:
rf_model = RandomForestClassifier(random_state=42)

grid_rf = GridSearchCV(
    rf_model,
    param_grid_rf,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)


,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'max_depth': [None, 5, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [100, 200, ...]}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


In [15]:
grid_rf.best_params_

{'max_depth': 5, 'min_samples_split': 10, 'n_estimators': 200}

In [16]:
grid_rf.best_score_

np.float64(0.8002160537177782)

In [17]:
# 1. En iyi modeli al
best_rf = grid_rf.best_estimator_

# 2. Tahminleri üret (Hem Train hem Test için)
y_pred_train_rf = best_rf.predict(X_train)
y_pred_test_rf = best_rf.predict(X_test)

y_proba_train_rf = best_rf.predict_proba(X_train)[:, 1]
y_proba_test_rf = best_rf.predict_proba(X_test)[:, 1]

# 3. Performans Metriklerini Hesapla ve Yazdır
print("--- EĞİTİM (TRAIN) PERFORMANSI ---")
print(f"Accuracy: {accuracy_score(y_train, y_pred_train_rf):.4f}")
print(f"ROC AUC:  {roc_auc_score(y_train, y_proba_train_rf):.4f}")

print("\n--- TEST PERFORMANSI ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_test_rf):.4f}")
print(f"ROC AUC:  {roc_auc_score(y_test, y_proba_test_rf):.4f}")

--- EĞİTİM (TRAIN) PERFORMANSI ---
Accuracy: 0.8157
ROC AUC:  0.8627

--- TEST PERFORMANSI ---
Accuracy: 0.7863
ROC AUC:  0.8108


### Model Performance After Hyperparameter Tuning

**GridSearchCV** was applied to systematically search for optimal hyperparameters for both models. Based on the results:

* **Random Forest:** Showed improved generalization performance after tuning. By optimizing parameters like `n_estimators`, `max_depth`, and `min_samples_split`, the model successfully captured more complex patterns in the data, leading to a significant boost in both **Accuracy** and **ROC AUC** scores.
* **Logistic Regression:** Exhibited marginal improvements. As a linear model, its capacity is relatively limited compared to ensemble methods, and the baseline parameters were already near-optimal for this specific feature set.

**Conclusion:** The ensemble nature of Random Forest allowed it to benefit more from the tuning process, making it the more robust choice for this classification task.

In [19]:
import joblib
joblib.dump(best_rf, "best_random_forest.pkl")


['best_random_forest.pkl']